# Chapter 29: Deep Learning Operations

## Learning Objectives\n- Run reproducible deep learning experiments\n- Apply AdamW + scheduler stack correctly\n- Design robust checkpoint/recovery flow\n- Optimize training throughput

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 29: Deep Learning Operations (Industry Standard)

## Why this chapter matters
Training deep models at scale requires systems discipline: reproducibility, observability, and efficient compute usage.

## Learning goals
1. Build reproducible training runs.
2. Use optimizer/scheduler stacks correctly.
3. Apply checkpointing and fault tolerance.
4. Evaluate deployment efficiency.

## Step 1: Reproducibility policy
- fixed seeds
- deterministic data split snapshots
- pinned dependency versions
- tracked config files for each run

## Step 2: Optimizer stack
Default production baseline:
- AdamW
- cosine decay schedule + warmup
- gradient clipping
- AMP (if framework supports)

## Step 3: Checkpoint strategy
Save:
- model weights
- optimizer state
- scheduler state
- epoch/step counters
- best validation metric

## Step 4: Performance optimization
- mixed precision
- gradient accumulation
- efficient dataloading
- profiling bottlenecks (CPU input vs GPU compute)

## Step 5: Serving and lifecycle
- batch vs real-time inference
- quantization or distillation
- post-deployment drift monitoring
- scheduled or event-driven retraining

## Assignment
Design a training runbook for one model in this repository including:
1. exact hyperparameter config format
2. checkpoint policy
3. rollback and retrain criteria



In [ ]:
from __future__ import annotations

import math
from typing import List


def cosine_lr(step: int, total_steps: int, warmup_steps: int, base_lr: float, min_lr: float) -> float:
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (base_lr - min_lr) * cosine


class AdamW1D:
    def __init__(self, n_params: int, lr: float = 1e-3, wd: float = 1e-2, b1: float = 0.9, b2: float = 0.999):
        self.lr = lr
        self.wd = wd
        self.b1 = b1
        self.b2 = b2
        self.eps = 1e-8
        self.m = [0.0] * n_params
        self.v = [0.0] * n_params
        self.t = 0

    def step(self, params: List[float], grads: List[float], lr: float) -> List[float]:
        self.t += 1
        for i in range(len(params)):
            g = grads[i]
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g * g
            m_hat = self.m[i] / (1 - self.b1 ** self.t)
            v_hat = self.v[i] / (1 - self.b2 ** self.t)
            params[i] -= lr * (m_hat / (math.sqrt(v_hat) + self.eps) + self.wd * params[i])
        return params


if __name__ == "__main__":
    params = [1.5, -2.0]
    target = [0.0, 0.0]
    opt = AdamW1D(n_params=2)

    total_steps = 200
    for step in range(total_steps):
        grads = [2 * (p - t) for p, t in zip(params, target)]
        lr = cosine_lr(step, total_steps, warmup_steps=20, base_lr=0.05, min_lr=0.001)
        params = opt.step(params, grads, lr)

        if step % 25 == 0:
            print(f"step={step:03d} lr={lr:.4f} params={[round(x, 4) for x in params]}")


## Checkpoint\n\n1. You can explain warmup + cosine decay behavior.\n2. You can describe AdamW decoupled weight decay.\n3. You can define minimum checkpoint contents.

## Guided Exercise\nPlot LR values across steps and verify warmup then cosine decay shape.

In [ ]:
# TODO (Student Exercise)
vals = [cosine_lr(s, 200, 20, 0.05, 0.001) for s in range(200)]
print(vals[:5], vals[20], vals[-1])

## Quick Quiz\n\n**Q1.** Why checkpoint optimizer state too?\n\n**Answer:** To resume with correct momentum/variance estimates.\n\n**Q2.** What is one benefit of mixed precision?\n\n**Answer:** Higher throughput and lower memory usage.\n\n**Q3.** When should retraining trigger?\n\n**Answer:** When monitored drift/performance crosses defined thresholds.\n

## Homework\nWrite a training runbook with config, checkpoint policy, and rollback criteria.